# Análise Preditiva de RH — TechNova Solutions
## Análise Exploratória de Dados (EDA) + Machine Learning

**Dataset:** IBM HR Employee Attrition & Performance (enriquecido com scores comportamentais)

**Objetivos:**
- Entender os fatores que levam ao desligamento (Attrition)
- Treinar e otimizar modelos preditivos com busca de hiperparâmetros (Optuna)
- Segmentar funcionários em perfis via K-Means com K escolhido pelos dados
- Exportar artefatos para uso no app Streamlit


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Seed global para reprodutibilidade total
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print("Bibliotecas carregadas com sucesso!")


Todas as bibliotecas necessárias para o projeto são carregadas aqui de uma vez.
A constante `RANDOM_STATE = 42` é definida globalmente e passada para todos os algoritmos
que envolvem aleatoriedade — isso garante que qualquer pessoa que execute este notebook
obtenha exatamente os mesmos resultados, tornando a análise **reproduzível**.

`warnings.filterwarnings('ignore')` suprime avisos de versão que poluem a saída
sem afetar o funcionamento do código.

---
## 1. Carregamento e Visão Geral dos Dados


In [ ]:
df = pd.read_csv('../data/hr_dataset.csv')

print(f"Shape: {df.shape}")
print(f"Registros: {df.shape[0]} | Colunas: {df.shape[1]}")
print(f"\nAttrition (desbalanceamento):")
print(df['Attrition'].value_counts())
print(f"Taxa de Attrition: {df['Attrition'].mean():.1%}")
df.head()


O dataset carregado é baseado no **IBM HR Employee Attrition & Performance** (disponível no Kaggle),
enriquecido com scores comportamentais gerados sinteticamente para simular avaliações de RH reais:
`SoftSkillsScore`, `TechnicalSkillsScore`, `CultureFitScore`, `EngagementScore` e outros.

A coluna alvo é `Attrition` (0 = ficou, 1 = saiu).
Repare na taxa de attrition impressa: ela costuma ficar em torno de **22%**,
o que significa que o dataset é **desbalanceado** — há muito mais registros de quem ficou
do que de quem saiu. Isso terá impacto direto nas escolhas de modelagem.

In [ ]:
print("--- Info do Dataset ---")
print(df.dtypes.value_counts())
print(f"\nValores nulos: {df.isnull().sum().sum()}")
print(f"\nEstatísticas descritivas (amostra):")
df.describe().round(2)


Esta célula faz a **auditoria de qualidade dos dados**:

- `dtypes.value_counts()` mostra quantas colunas são numéricas e quantas são categóricas (texto),
  o que determina quais precisarão de encoding antes do ML.
- `isnull().sum().sum()` verifica se há dados faltantes. Com zero nulos, não precisamos
  de imputação — uma etapa a menos de pré-processamento.
- `describe()` exibe média, desvio padrão, mínimo e máximo de cada coluna numérica,
  permitindo identificar rapidamente valores fora do esperado ou escalas muito diferentes
  entre as features.

---
## 2. Análise de Attrition (Desligamento)
Entender quais fatores mais influenciam a saída de funcionários.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Fatores de Attrition', fontsize=16, fontweight='bold')

# OverTime
att_ot = df.groupby('OverTime')['Attrition'].mean()
axes[0,0].bar(att_ot.index, att_ot.values, color=['#059669','#dc2626'])
axes[0,0].set_title('Attrition por Hora Extra')
axes[0,0].set_ylabel('Taxa de Attrition')
axes[0,0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))

# JobSatisfaction
att_js = df.groupby('JobSatisfaction')['Attrition'].mean()
axes[0,1].bar(att_js.index, att_js.values, color=sns.color_palette('YlOrRd', 4))
axes[0,1].set_title('Attrition por Satisfação no Trabalho')
axes[0,1].set_xlabel('Nível de Satisfação (1=baixo, 4=alto)')
axes[0,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))

# Age
axes[0,2].hist([df[df['Attrition']==0]['Age'], df[df['Attrition']==1]['Age']],
               bins=20, label=['Ficou','Saiu'], color=['#059669','#dc2626'], alpha=0.7)
axes[0,2].set_title('Distribuição de Idade por Attrition')
axes[0,2].legend()

# MonthlyIncome
axes[1,0].boxplot([df[df['Attrition']==0]['MonthlyIncome'],
                   df[df['Attrition']==1]['MonthlyIncome']],
                  labels=['Ficou','Saiu'])
axes[1,0].set_title('Salário Mensal vs Attrition')
axes[1,0].set_ylabel('Salário (R$)')

# YearsAtCompany
att_yac = df.groupby('YearsAtCompany')['Attrition'].mean().head(20)
axes[1,1].bar(att_yac.index, att_yac.values, color='steelblue', alpha=0.8)
axes[1,1].set_title('Attrition por Anos na Empresa')
axes[1,1].set_xlabel('Anos na Empresa')
axes[1,1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0%}'))

# Department
att_dept = df.groupby('Department')['Attrition'].mean().sort_values(ascending=False)
axes[1,2].barh(att_dept.index, att_dept.values,
               color=sns.color_palette('husl', len(att_dept)))
axes[1,2].set_title('Attrition por Departamento')
axes[1,2].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.0%}'))

plt.tight_layout()
plt.savefig('../assets/attrition_factors.png', dpi=150, bbox_inches='tight')
plt.show()


Os seis gráficos exploram as variáveis que a literatura de RH aponta como
mais associadas ao desligamento voluntário:

- **Hora extra (OverTime):** funcionários que fazem hora extra sistematicamente
  apresentam taxa de saída muito maior — indício de esgotamento e desequilíbrio
  entre vida pessoal e profissional.
- **Satisfação no trabalho (JobSatisfaction):** escala de 1 a 4; quanto menor,
  maior a propensão a sair. A queda entre os níveis 1 e 2 costuma ser a mais acentuada.
- **Idade:** profissionais mais jovens trocam de emprego com mais frequência,
  especialmente nos primeiros anos de carreira.
- **Salário:** a caixa do boxplot de quem saiu tende a estar abaixo da de quem ficou,
  confirmando que remuneração abaixo do mercado é fator de risco.
- **Anos na empresa:** os primeiros 1 a 3 anos concentram o maior risco de saída —
  período em que o profissional ainda está avaliando se a empresa é o lugar certo.
- **Departamento:** permite identificar áreas com gestão ou cultura mais problemática.

Esses padrões guiam a interpretação dos modelos nas próximas seções.

---
## 3. Análise de Correlações


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

corr_att = corr_matrix['Attrition'].drop('Attrition').sort_values(key=abs, ascending=False).head(15)
print("Top 15 correlações com Attrition:")
print(corr_att.round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdYlGn',
            center=0, ax=axes[0], linewidths=0.1)
axes[0].set_title('Matriz de Correlação Completa', fontweight='bold')

colors = ['#dc2626' if v > 0 else '#059669' for v in corr_att.values]
axes[1].barh(corr_att.index, corr_att.values, color=colors)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Top Correlações com Attrition', fontweight='bold')
axes[1].set_xlabel('Correlação de Pearson')

plt.tight_layout()
plt.savefig('../assets/correlations.png', dpi=150, bbox_inches='tight')
plt.show()



A **correlação de Pearson** mede a relação linear entre cada variável numérica e o alvo `Attrition`.
Ela varia de -1 a +1:

- **Valor positivo (barras vermelhas):** quanto maior a variável, maior a chance de sair.
  Ex.: `OverTime` codificado como 0/1 — ter hora extra aumenta o risco.
- **Valor negativo (barras verdes):** quanto maior a variável, menor a chance de sair.
  Ex.: `JobSatisfaction` alto está associado a permanecer na empresa.

Em dados de RH, correlações acima de **|0.15|** já são consideradas relevantes para negócio,
mesmo que modestas do ponto de vista estatístico puro.

O heatmap da esquerda mostra a correlação entre **todas** as variáveis entre si,
útil para detectar multicolinearidade — quando duas features carregam informação redundante,
o que pode afetar modelos lineares como a Regressão Logística.


---
## 4. Análise dos Scores Enriquecidos por Departamento


In [ ]:
score_cols = ['SoftSkillsScore', 'TechnicalSkillsScore', 'LeadershipScore',
              'CommunicationScore', 'AdaptabilityScore', 'InnovationScore',
              'TeamworkScore', 'CultureFitScore', 'EngagementScore', 'GrowthPotential']

dept_scores = df.groupby('Department')[score_cols].mean()

fig, ax = plt.subplots(figsize=(14, 6))
dept_scores.T.plot(kind='bar', ax=ax, colormap='Set2')
ax.set_title('Scores Médios por Departamento', fontweight='bold', fontsize=14)
ax.set_ylabel('Score Médio (0–10)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(title='Departamento', bbox_to_anchor=(1.01, 1))
ax.set_ylim(0, 10)
plt.tight_layout()
plt.savefig('../assets/department_scores.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMédia geral por score (decrescente):")
print(df[score_cols].mean().round(2).sort_values(ascending=False).to_string())


Esta análise responde à pergunta: **os scores comportamentais variam entre departamentos?**

Se um departamento apresenta sistematicamente scores de `CultureFitScore` ou `EngagementScore`
mais baixos que os demais, isso é um sinal de alerta para o RH — pode indicar problemas
de liderança, cultura local ou inadequação dos processos de seleção naquela área.

A tabela impressa abaixo do gráfico mostra a média geral de cada score em ordem decrescente,
permitindo identificar quais dimensões comportamentais estão mais desenvolvidas
(ou precisam de mais atenção) na empresa como um todo.

---
## 5. Distribuições e Outliers dos Scores


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 10))
fig.suptitle('Distribuição dos Scores — Ficou vs Saiu', fontsize=16, fontweight='bold')

for i, col in enumerate(score_cols):
    ax = axes[i//5, i%5]
    ax.hist(df[df['Attrition']==0][col], bins=15, alpha=0.6,
            label='Ficou', color='#059669', density=True)
    ax.hist(df[df['Attrition']==1][col], bins=15, alpha=0.6,
            label='Saiu',  color='#dc2626', density=True)
    ax.set_title(col.replace('Score','').replace('Skills',' Skills'), fontsize=10)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('../assets/score_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


Cada histograma sobrepõe a distribuição de um score para quem ficou (verde)
e para quem saiu (vermelho), com densidade normalizada para que a comparação
seja justa independentemente do tamanho de cada grupo.

Quando as duas curvas se separam claramente — por exemplo, `CultureFitScore`
e `EngagementScore` tendendo a valores mais baixos entre quem saiu —
isso confirma que aquela feature tem **poder discriminativo real**
e provavelmente aparecerá entre as mais importantes nos modelos de ML.

Quando as curvas se sobrepõem muito, a feature contribui pouco individualmente
(embora possa ser útil em combinação com outras dentro de um modelo não-linear).

---
## 6. Preparação dos Dados para Machine Learning

Etapas:
1. **Encoding** de variáveis categóricas com `LabelEncoder`
2. **Split estratificado** treino/teste — preserva a proporção de Attrition em ambos os conjuntos
3. **SMOTE** aplicado apenas no treino para balancear a classe minoritária
4. **StandardScaler** ajustado somente no treino e aplicado no teste (evita data leakage)


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, accuracy_score, f1_score,
                             precision_recall_curve, average_precision_score)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib

# Encoding das variáveis categóricas
df_ml   = df.copy()
le_dict = {}
cat_cols = df_ml.select_dtypes(include=['object', 'string']).columns

for col in cat_cols:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[col])
    le_dict[col] = le

print(f"Colunas categóricas encodadas: {list(cat_cols)}")
print(f"Shape após encoding: {df_ml.shape}")

# Features e target
feature_cols = [c for c in df_ml.columns if c not in ['EmployeeID', 'Attrition']]
X = df_ml[feature_cols]
y = df_ml['Attrition']

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"\nTreino : {y_train.value_counts().to_dict()}")
print(f"Teste  : {y_test.value_counts().to_dict()}")
print("\nA avaliação a seguir usa apenas o conjunto de treino para CV, threshold e tuning.")


Esta célula faz toda a **preparação dos dados** para os modelos de ML:

**1. Label Encoding**
Colunas com texto (ex.: `Department = "Sales"`, `Gender = "Male"`) são convertidas
para números inteiros, pois os algoritmos de ML operam exclusivamente com valores numéricos.
Um `LabelEncoder` separado é salvo para cada coluna — isso é necessário para o app Streamlit
poder transformar dados novos da mesma forma que o modelo aprendeu.

**2. Split estratificado (80% treino / 20% teste)**
O parâmetro `stratify=y` garante que a proporção de attrition (≈22%)
seja mantida em **ambos** os conjuntos. Sem isso, por azar o conjunto de teste
poderia ter muito mais ou muito menos casos de saída do que o real,
distorcendo a avaliação.

**Regra de ouro que seguimos:** o conjunto de teste não será tocado até a avaliação final.
Todo o tuning, escolha de threshold e validação cruzada acontece **apenas nos dados de treino**.

---
## 7. Treinamento e Otimização dos Modelos de Classificação

Estratégias de avaliação robusta aplicadas:
- **Optuna** com `SMOTE` dentro do pipeline para benchmarking honesto do XGBoost
- **StratifiedKFold (5 folds)** para gerar predições out-of-fold no treino
- **Threshold customizado** ajustado apenas no treino, via curva Precision-Recall out-of-fold
- Comparação justa entre modelos para entender generalização e estabilidade
- Escolha do melhor modelo por **CV F1 (OOF)** dentro do pipeline atual


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def build_pipeline(model, use_scaler=False):
    steps = []
    if use_scaler:
        steps.append(('scaler', StandardScaler()))
    steps.append(('smote', SMOTE(random_state=RANDOM_STATE)))
    steps.append(('model', model))
    return ImbPipeline(steps)


def find_best_threshold(y_true, y_proba):
    prec, rec, thresholds = precision_recall_curve(y_true, y_proba)
    f1_curve = 2 * prec * rec / (prec + rec + 1e-9)
    best_idx = int(np.argmax(f1_curve[:-1]))
    return float(thresholds[best_idx]), float(f1_curve[:-1][best_idx])


# ── Otimização bayesiana honesta dos hiperparâmetros do XGBoost ──────────────
def objective(trial):
    params = {
        'n_estimators'    : trial.suggest_int('n_estimators', 100, 400),
        'max_depth'       : trial.suggest_int('max_depth', 3, 8),
        'learning_rate'   : trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample'       : trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma'           : trial.suggest_float('gamma', 0, 5),
        'reg_alpha'       : trial.suggest_float('reg_alpha', 0, 1),
        'reg_lambda'      : trial.suggest_float('reg_lambda', 0.5, 2),
        'eval_metric'     : 'logloss',
        'verbosity'       : 0,
        'random_state'    : RANDOM_STATE,
    }
    pipeline = build_pipeline(XGBClassifier(**params))
    oof_proba = cross_val_predict(
        pipeline, X_train, y_train, cv=cv, method='predict_proba', n_jobs=1
    )[:, 1]
    threshold, f1 = find_best_threshold(y_train, oof_proba)
    trial.set_user_attr('best_threshold', threshold)
    trial.set_user_attr('cv_auc', float(roc_auc_score(y_train, oof_proba)))
    trial.set_user_attr('cv_ap', float(average_precision_score(y_train, oof_proba)))
    return f1


print("Otimizando XGBoost com Optuna e avaliação out-of-fold honesta (20 trials)...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=False)
best_xgb_params = study.best_params
best_xgb_params.update({'eval_metric': 'logloss', 'verbosity': 0, 'random_state': RANDOM_STATE})
print(f"Melhores hiperparâmetros XGBoost encontrados (F1 OOF = {study.best_value:.4f}):")
for k, v in best_xgb_params.items():
    if k not in ('eval_metric','verbosity','random_state'):
        print(f"  {k}: {v}")
print(f"Threshold OOF do melhor trial: {study.best_trial.user_attrs['best_threshold']:.3f}")
print(f"CV AUC do melhor trial       : {study.best_trial.user_attrs['cv_auc']:.4f}")
print(f"CV AP do melhor trial        : {study.best_trial.user_attrs['cv_ap']:.4f}")


Esta célula define a **infraestrutura de avaliação** usada em todo o restante do notebook.

**`build_pipeline(model, use_scaler)`**
Monta um pipeline que encadeia, nessa ordem: (opcional) StandardScaler → SMOTE → modelo.
O ponto crítico é que o SMOTE está **dentro** do pipeline: ao fazer validação cruzada,
o balanceamento acontece apenas nos dados de treino de cada fold, nunca vazando
informação para o fold de validação. Fazer SMOTE antes da CV é um erro clássico
que infla artificialmente as métricas.

**`find_best_threshold(y_true, y_proba)`**
O padrão de 0.5 como ponto de corte foi projetado para classes equilibradas.
Em dados com 22% de attrition, usar 0.5 faz o modelo ser conservador demais
na detecção de quem vai sair. Esta função varre a curva Precision-Recall
e escolhe o threshold que maximiza o **F1-Score** — o equilíbrio entre
capturar quem vai sair (recall) sem gerar alertas falsos demais (precision).

**Optuna — otimização bayesiana do XGBoost**
Em vez de testar todas as combinações possíveis de hiperparâmetros (grid search),
o Optuna aprende com os trials anteriores para focar nas regiões mais promissoras
do espaço de busca. Cada trial avalia o XGBoost usando predições **out-of-fold (OOF)**
no conjunto de treino, mantendo o teste completamente isolado durante a otimização.
Os 9 hiperparâmetros ajustados controlam a complexidade do modelo (`max_depth`, `gamma`),
a taxa de aprendizado, a regularização (`reg_alpha`, `reg_lambda`) e a amostragem
de dados e features por árvore (`subsample`, `colsample_bytree`).

In [ ]:


# ── Treinar e comparar todos os modelos com avaliação honesta ────────────────
model_specs = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, C=0.5, random_state=RANDOM_STATE),
        'use_scaler': True, 
    },
    'Random Forest': {
        'model': RandomForestClassifier(
            n_estimators=300, max_depth=12, min_samples_leaf=2, random_state=RANDOM_STATE
        ),
        'use_scaler': False,
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8,
            random_state=RANDOM_STATE
        ),
        'use_scaler': False,
    },
    'XGBoost (Optuna)': {
        'model': XGBClassifier(**best_xgb_params),
        'use_scaler': False,
    },
}

results = {}
print()
for name, spec in model_specs.items():
    pipeline = build_pipeline(spec['model'], use_scaler=spec['use_scaler'])

    # Predições out-of-fold no treino: base para CV honesta e escolha do threshold
    oof_proba = cross_val_predict(
        pipeline, X_train, y_train, cv=cv, method='predict_proba', n_jobs=1
    )[:, 1]
    best_thr, cv_f1 = find_best_threshold(y_train, oof_proba)
    oof_pred = (oof_proba >= best_thr).astype(int)

    cv_auc = roc_auc_score(y_train, oof_proba)
    cv_ap = average_precision_score(y_train, oof_proba)

    # Ajuste final no treino completo e avaliação única no teste
    pipeline.fit(X_train, y_train)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= best_thr).astype(int)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)

    results[name] = {
        'accuracy': acc,
        'f1_score': f1,
        'auc_roc': auc,
        'avg_precision': ap,
        'cv_f1_oof': cv_f1,
        'cv_auc_oof': cv_auc,
        'cv_ap_oof': cv_ap,
        'oof_pred': oof_pred,
        'oof_proba': oof_proba,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'model': pipeline,
        'threshold': best_thr,
    }

    print(f"{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"  CV OOF  : F1 = {cv_f1:.4f} | AUC = {cv_auc:.4f} | AP = {cv_ap:.4f}")
    print(f"  Teste   : Thr = {best_thr:.3f} | Acc = {acc:.4f} | F1 = {f1:.4f} | AUC = {auc:.4f} | AP = {ap:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Ficou', 'Saiu']))

best_name = max(results, key=lambda x: results[x]['cv_f1_oof'])
selected_model_name = best_name
selected_result = results[selected_model_name]

print(f"\nMelhor modelo por CV F1 (OOF): {selected_model_name}")
print(f"  Threshold operacional = {selected_result['threshold']:.3f}")
print(f"  Test F1 = {selected_result['f1_score']:.4f}")
print(f"  Test AUC= {selected_result['auc_roc']:.4f}")
print(f"  Test AP = {selected_result['avg_precision']:.4f}")



Quatro modelos são treinados e avaliados com o mesmo protocolo rigoroso:

| Modelo | Característica principal |
|---|---|
| Logistic Regression | Linear, interpretável, sensível à escala (usa StandardScaler) |
| Random Forest | Ensemble de árvores independentes, robusto a outliers |
| Gradient Boosting | Ensemble sequencial, corrige erros incrementalmente |
| XGBoost (Optuna) | Gradient boosting otimizado com hiperparâmetros calibrados |

**Fluxo de avaliação para cada modelo:**
1. `cross_val_predict` gera probabilidades **out-of-fold** no treino completo —
   cada exemplo é avaliado pelo modelo que *não* o viu durante o treino.
2. `find_best_threshold` encontra o melhor ponto de corte com base nessas probabilidades OOF.
3. O pipeline é retreinado no treino **completo** (sem divisão de folds).
4. O modelo retreinado é avaliado **uma única vez** no conjunto de teste.

Esse fluxo evita dois erros comuns: escolher o threshold olhando para o teste
(contaminação) e reportar métricas do CV como se fossem métricas de teste (otimismo).

O **melhor modelo é selecionado pelo CV F1 OOF** — não pelo resultado no teste —
para que a escolha seja baseada em dados que o modelo não usou para se ajustar.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Comparação dos Modelos', fontsize=16, fontweight='bold')

# 1. Métricas comparativas
metrics_df = pd.DataFrame({
    name: {
        'CV F1 (OOF)': r['cv_f1_oof'],
        'CV AUC (OOF)': r['cv_auc_oof'],
        'Teste F1': r['f1_score'],
        'Teste AP': r['avg_precision'],
    }
    for name, r in results.items()
}).T
metrics_df.plot(kind='bar', ax=axes[0,0], colormap='viridis', edgecolor='white')
axes[0,0].set_title('Métricas Comparativas', fontweight='bold')
axes[0,0].set_ylim(0, 1)
axes[0,0].set_xticklabels(axes[0,0].get_xticklabels(), rotation=35, ha='right')
axes[0,0].legend(loc='lower right', fontsize=9)
for bar in axes[0,0].patches:
    h = bar.get_height()
    if h > 0.05:
        axes[0,0].text(bar.get_x()+bar.get_width()/2, h+0.01,
                       f'{h:.2f}', ha='center', va='bottom', fontsize=7)

# 2. Curvas ROC
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_proba'])
    axes[0,1].plot(fpr, tpr, label=f"{name} ({r['auc_roc']:.3f})", linewidth=2)
axes[0,1].plot([0,1],[0,1],'k--', alpha=0.4)
axes[0,1].set_title('Curvas ROC (teste)', fontweight='bold')
axes[0,1].set_xlabel('False Positive Rate')
axes[0,1].set_ylabel('True Positive Rate')
axes[0,1].legend(fontsize=9)

# 3. Curvas Precision-Recall
for name, r in results.items():
    prec, rec, _ = precision_recall_curve(y_test, r['y_proba'])
    axes[1,0].plot(rec, prec, label=f"{name} (AP={r['avg_precision']:.3f})", linewidth=2)
baseline = y_test.mean()
axes[1,0].axhline(baseline, color='gray', linestyle='--',
                   label=f'Baseline aleatório ({baseline:.2f})')
axes[1,0].set_title('Curvas Precision-Recall (teste)', fontweight='bold')
axes[1,0].set_xlabel('Recall')
axes[1,0].set_ylabel('Precision')
axes[1,0].legend(fontsize=9)

# 4. Confusion Matrix do modelo escolhido dinamicamente
cm = confusion_matrix(y_test, selected_result['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1,1],
            xticklabels=['Ficou','Saiu'], yticklabels=['Ficou','Saiu'])
axes[1,1].set_title(f'Confusion Matrix — {selected_model_name}', fontweight='bold')
axes[1,1].set_ylabel('Real')
axes[1,1].set_xlabel('Previsto')

plt.tight_layout()
plt.savefig('../assets/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMelhor modelo por CV F1 (OOF): {selected_model_name}")
print(f"  CV F1 = {selected_result['cv_f1_oof']:.4f}")
print(f"  CV AUC= {selected_result['cv_auc_oof']:.4f}")
print(f"  Test F1 = {selected_result['f1_score']:.4f}")
print(f"  Test AUC= {selected_result['auc_roc']:.4f}")
print(f"  Test AP = {selected_result['avg_precision']:.4f}")
print(f"  Threshold = {selected_result['threshold']:.4f}")


Os quatro gráficos apresentam perspectivas complementares sobre o desempenho dos modelos:

**Métricas comparativas (esquerda superior)**
Exibe quatro métricas lado a lado para cada modelo. As métricas de CV (OOF) mostram
o desempenho estimado durante a otimização; as de teste mostram o resultado real.
Modelos com CV e teste próximos generalizam bem — modelos com CV muito acima do teste
estão sofrendo overfitting.

**Curvas ROC (direita superior)**
Mostram o trade-off entre taxa de verdadeiros positivos (quem saiu detectado corretamente)
e falsos positivos (quem ficou classificado erroneamente como saindo).
Área sob a curva (AUC) mais próxima de 1.0 é melhor.
A linha tracejada diagonal representa um classificador completamente aleatório (AUC = 0.5).

**Curvas Precision-Recall (esquerda inferior)**
Mais informativas que a ROC para dados desbalanceados.
A linha tracejada horizontal é o **baseline**: um classificador que sempre diz "vai sair"
teria precisão igual à taxa de attrition real (≈22%).
Quanto maior a área sob a curva (Average Precision), melhor o modelo identifica
casos de saída sem gerar alertas desnecessários.

**Matriz de Confusão do modelo vencedor (direita inferior)**
Mostra os quatro tipos de resultado possíveis:
- **Verdadeiros Negativos (topo-esquerda):** ficou e foi previsto que ficaria — correto
- **Falsos Positivos (topo-direita):** ficou mas foi previsto que sairia — alerta desnecessário
- **Falsos Negativos (baixo-esquerda):** saiu mas foi previsto que ficaria — o mais custoso
- **Verdadeiros Positivos (baixo-direita):** saiu e foi previsto que sairia — correto

Em RH, **Falsos Negativos são o erro mais caro**: deixar de identificar alguém
que vai sair significa perder a oportunidade de agir com antecedência.
O threshold customizado foi escolhido justamente para reduzir esse tipo de erro.

O modelo baseado em XGBoost otimizado com Optuna apresentou o melhor desempenho geral, com F1-score de 0.41 no conjunto de teste. A escolha de um threshold mais baixo (0.26) prioriza a identificação de colaboradores com risco de saída, aumentando o recall, porém com impacto na precisão, resultando em maior número de falsos positivos.
Após otimizações e ajuste de threshold, o modelo passou a capturar aproximadamente 67% dos casos de churn, tornando-se significativamente mais útil para aplicações de negócio, apesar do aumento de falsos positivos.”


In [ ]:
best_model = results[best_name]['model']

# 🔥 pega o XGBoost dentro do pipeline
if hasattr(best_model, 'named_steps'):
    model = best_model.named_steps['model']
else:
    model = best_model

# 🔥 agora funciona com XGBClassifier
if hasattr(model, 'feature_importances_'):

    importance = pd.Series(model.feature_importances_,
                           index=feature_cols).sort_values(ascending=True)

    top20 = importance.tail(20)

    fig, ax = plt.subplots(figsize=(10, 12))

    colors = ['#059669' if i >= len(top20)-5 else 'steelblue'
              for i in range(len(top20))]

    top20.plot(kind='barh', ax=ax, color=colors)

    ax.set_title(f'Top 20 Features Mais Importantes — {best_name}',
                 fontsize=14, fontweight='bold')

    ax.set_xlabel('Importância')
    ax.axvline(top20.mean(), color='red', linestyle='--',
               alpha=0.6, label='Média')

    ax.legend()

    plt.tight_layout()
    plt.savefig('../assets/feature_importance.png',
                dpi=150, bbox_inches='tight')
    plt.show()

    # 🔝 Top 5
    top5 = importance.tail(5).index.tolist()[::-1]

    print("Top 5 features mais importantes:")
    for i, f in enumerate(top5, 1):
        print(f"  {i}. {f} ({importance[f]:.4f})")

else:
    print(f"Modelo {best_name} não suporta feature importance.")

O gráfico mostra as 20 variáveis que mais influenciaram as decisões do modelo vencedor,
medidas pelo **ganho de informação** médio que cada feature proporciona ao dividir os dados
nas árvores de decisão internas do XGBoost.

As **5 features destacadas em verde** são as mais importantes e merecem atenção especial:
elas são os fatores onde intervenções de RH terão maior impacto na retenção de talentos.

A **linha vermelha tracejada** marca a importância média. Features acima dela contribuem
acima da média para o modelo. Features muito abaixo da linha carregam pouca informação
adicional e poderiam ser removidas em uma versão futura sem grande perda de desempenho.

Note que o código extrai o modelo do interior do pipeline (`named_steps['model']`)
antes de acessar `feature_importances_` — isso é necessário porque o objeto salvo
é o pipeline completo, não o modelo isolado.

---
## 8. Clustering com K-Means — Perfis de Funcionários




In [ ]:
from sklearn.metrics import silhouette_score

cluster_features = ['Age', 'MonthlyIncome', 'YearsAtCompany', 'TotalWorkingYears',
                    'JobSatisfaction', 'WorkLifeBalance', 'PerformanceRating',
                    'SoftSkillsScore', 'TechnicalSkillsScore', 'LeadershipScore',
                    'CommunicationScore', 'AdaptabilityScore', 'InnovationScore',
                    'TeamworkScore', 'CultureFitScore', 'EngagementScore',
                    'GrowthPotential']

X_cluster        = df_ml[cluster_features].copy()
scaler_cluster   = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

K_range    = range(2, 11)
inertias   = []
sil_scores = []

for k in K_range:
    km     = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_cluster_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster_scaled, labels))

best_k = K_range[np.argmax(sil_scores)]
print(f"K ideal pelo Silhouette Score: {best_k}  (score = {max(sil_scores):.4f})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_title('Método do Cotovelo (Elbow Method)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inércia')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, sil_scores, 'ro-', linewidth=2, markersize=8)
axes[1].axvline(x=best_k, color='green', linestyle='--', linewidth=2,
                label=f'K ideal = {best_k}')
axes[1].set_title('Silhouette Score por K', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig('../assets/elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()


O K-Means precisa que o número de clusters (K) seja definido antes de rodar.
Aqui usamos dois critérios complementares para escolher K com base nos dados,
sem arbitrariedade:

**Elbow Method (gráfico da esquerda)**
A inércia mede a soma das distâncias quadradas de cada ponto ao centróide do seu cluster.
Ela sempre cai conforme K aumenta — o que importa é identificar o ponto onde
a queda passa a ser marginal, formando um "cotovelo" na curva.
Após esse ponto, adicionar mais clusters gera grupos muito pequenos e pouco significativos.

**Silhouette Score (gráfico da direita)**
Mede o quão bem cada ponto se encaixa no seu próprio cluster **comparado** aos outros clusters.
Varia de -1 a +1:
- Próximo de +1: o ponto está claramente no cluster certo
- Próximo de 0: o ponto está na fronteira entre dois clusters
- Negativo: o ponto provavelmente está no cluster errado

O K com o maior Silhouette Score é selecionado automaticamente como `best_k`.
Isso torna a escolha **reproduzível e imparcial** — o K não foi escolhido manualmente
para dar um resultado mais bonito ou fazer sentido intuitivo.

In [ ]:
# K determinado pelos dados — não hardcoded
K_FINAL = best_k
print(f"Aplicando K-Means com K={K_FINAL} (definido pelo Silhouette Score)")

kmeans = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init='auto')
df['Cluster'] = kmeans.fit_predict(X_cluster_scaled)
df_ml['Cluster'] = df['Cluster']

# PCA para visualização 2D
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_cluster_scaled)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

scatter = axes[0].scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=df['Cluster'], cmap='Set1', alpha=0.6, s=30
)

axes[0].set_title(
    f'Clusters de Funcionários — PCA 2D  (K={K_FINAL})',
    fontsize=14, fontweight='bold'
)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variância)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variância)")
plt.colorbar(scatter, ax=axes[0], label='Cluster')

# Heatmap
cluster_profile = df.groupby('Cluster')[cluster_features].mean()
cluster_profile_norm = (
    (cluster_profile - cluster_profile.min()) /
    (cluster_profile.max() - cluster_profile.min() + 1e-9)
)

sns.heatmap(
    cluster_profile_norm.T,
    annot=cluster_profile.T.round(1).values,
    fmt='',
    cmap='YlOrRd',
    ax=axes[1]
)

axes[1].set_title(
    'Perfil Médio por Cluster (normalizado)',
    fontsize=14, fontweight='bold'
)
axes[1].set_xlabel('Cluster')

plt.tight_layout()
plt.savefig('../assets/clusters.png', dpi=150, bbox_inches='tight')
plt.show()

# 🔥 Nomeação automática baseada em ranking (corrigido)
cluster_summary = df.groupby('Cluster').agg({
    'EngagementScore': 'mean',
    'TechnicalSkillsScore': 'mean',
    'LeadershipScore': 'mean',
    'GrowthPotential': 'mean'
})

# Score composto
cluster_summary['score'] = (
    cluster_summary['EngagementScore'] * 0.3 +
    cluster_summary['GrowthPotential'] * 0.3 +
    cluster_summary['TechnicalSkillsScore'] * 0.2 +
    cluster_summary['LeadershipScore'] * 0.2
)

# Ordena clusters do melhor para o pior
ranked = cluster_summary.sort_values('score', ascending=False)

cluster_names = {}

for i, c in enumerate(ranked.index):
    if i == 0:
        cluster_names[c] = "Alto Potencial"
    elif i == len(ranked) - 1:
        cluster_names[c] = "Em Desenvolvimento"
    else:
        cluster_names[c] = "Intermediário"

# Exibir resultados
print()
for c in range(K_FINAL):
    sub = df[df['Cluster'] == c]

    print(f"{'='*60}")
    print(f"  CLUSTER {c} — {cluster_names[c]}  ({len(sub)} funcionários · {len(sub)/len(df):.0%})")
    print(f"{'='*60}")
    print(f"  Idade média       : {sub['Age'].mean():.0f} anos")
    print(f"  Salário médio     : R$ {sub['MonthlyIncome'].mean():,.0f}")
    print(f"  Anos na empresa   : {sub['YearsAtCompany'].mean():.1f}")
    print(f"  Culture Fit       : {sub['CultureFitScore'].mean():.1f}/10")
    print(f"  Engajamento       : {sub['EngagementScore'].mean():.1f}/10")
    print(f"  Potencial         : {sub['GrowthPotential'].mean():.1f}/10")
    print(f"  Taxa de Attrition : {sub['Attrition'].mean():.1%}")

Com o K ideal determinado, o K-Means final é treinado e cada funcionário
recebe a etiqueta do seu cluster.

**Visualização PCA 2D (gráfico da esquerda)**
As 17 dimensões usadas no clustering são comprimidas em 2 componentes principais (PC1 e PC2)
para que possamos visualizar os grupos em um plano. A porcentagem indicada em cada eixo
mostra quanta variância original é preservada naquela dimensão.
É normal que os clusters pareçam sobrepostos: o PCA perde informação ao comprimir,
mas os grupos são bem separados no espaço original de 17 dimensões.

**Heatmap de perfil médio por cluster (gráfico da direita)**
Os valores anotados dentro do heatmap são as médias reais de cada feature por cluster.
A cor (normalizada de 0 a 1 dentro de cada linha) permite comparar visualmente
quais clusters se destacam em cada dimensão.

**Nomeação automática dos clusters**
Em vez de usar nomes fixos ou thresholds rígidos, os clusters são nomeados dinamicamente
com base em um score composto calculado a partir das principais dimensões (engajamento,
potencial de crescimento, habilidades técnicas e liderança).

Os clusters são então ordenados de forma relativa entre si:
- O cluster com melhor desempenho geral é classificado como **Alto Potencial**
- O cluster com pior desempenho geral é classificado como **Em Desenvolvimento**
- Clusters intermediários (quando K > 2) são classificados como **Intermediário**

Essa abordagem garante que os nomes reflitam diferenças reais entre os grupos,
independentemente da escala dos dados ou do valor de K escolhido, mantendo consistência
entre o notebook e o app em Streamlit.

---
## 9. Salvando Todos os Artefatos
Exportação de todos os objetos necessários para o app Streamlit.


In [ ]:
import os
os.makedirs('../models', exist_ok=True)

joblib.dump(selected_result['model'], '../models/attrition_model.joblib')
joblib.dump(selected_result['threshold'], '../models/attrition_threshold.joblib')
print(f"Modelo de Attrition salvo  : {selected_model_name}")
print(f"Threshold salvo           : {selected_result['threshold']:.3f}")

joblib.dump(kmeans,          '../models/kmeans_model.joblib')
joblib.dump(scaler_cluster,  '../models/scaler_cluster.joblib')
joblib.dump(le_dict,         '../models/label_encoders.joblib')
joblib.dump(feature_cols,    '../models/feature_cols.joblib')
joblib.dump(cluster_features,'../models/cluster_features.joblib')
joblib.dump(pca,             '../models/pca_model.joblib')
joblib.dump(cluster_names,   '../models/cluster_names.joblib')

print("\nTodos os artefatos salvos em ../models/")
print()
for f in sorted(os.listdir('../models')):
    size = os.path.getsize(f'../models/{f}') / 1024
    print(f"  {f:<45} {size:6.1f} KB")


Todos os objetos treinados são serializados com `joblib` e salvos na pasta `../models/`.
Cada arquivo tem uma responsabilidade específica no app Streamlit:

| Arquivo | O que contém | Para que serve no app |
|---|---|---|
| `attrition_model.joblib` | Pipeline completo (SMOTE + modelo vencedor) | Prever risco de saída do candidato |
| `attrition_threshold.joblib` | Ponto de corte calibrado | Classificar probabilidade em risco Alto/Médio/Baixo |
| `kmeans_model.joblib` | Modelo K-Means treinado | Identificar o perfil/cluster do candidato |
| `scaler_cluster.joblib` | StandardScaler das features de cluster | Normalizar dados novos antes do K-Means |
| `label_encoders.joblib` | Dicionário de LabelEncoders por coluna | Transformar categorias do candidato em números |
| `feature_cols.joblib` | Lista de colunas do modelo de attrition | Garantir que o app passe as features na ordem certa |
| `cluster_features.joblib` | Lista de colunas do clustering | Idem para o K-Means |
| `pca_model.joblib` | PCA treinado (2 componentes) | Visualização futura no app |
| `cluster_names.joblib` | Dicionário cluster → nome | Exibir o nome do perfil no resultado da entrevista |

O `attrition_threshold.joblib` é um artefato novo em relação à versão anterior:
ele garante que o app use exatamente o mesmo ponto de corte que foi calibrado
durante o treinamento, sem precisar hardcodar o valor 0.5.